> **Public reproducibility copy.** Notebook outputs and execution counts were cleared before public packaging to avoid local-path/account leakage. The scientific code and configuration cells are preserved. See the repository README for required data paths and reproducibility scope.

# Freeze AMD Dataset 594 — Production v1

Notebook ini membuat rilis dataset yang dibekukan dan tidak boleh diubah:

```text
AMD_DATASET_594_PRODUCTION_v1/
├── RAW_594/
├── PREPROCESSED_594/
├── ROI_594/
├── provenance/
├── outer_fold_assignments.csv
├── RELEASE_FILE_CHECKSUMS_SHA256.csv
├── RELEASE_METADATA.json
└── README.md
```

## Sebelum menjalankan

Sesuaikan:

```python
V3_OUTPUT_DIR = Path(r"V3_normal_provenance_production_v3")
RELEASE_DIR = Path(r"AMD_DATASET_594_PRODUCTION_v1")
```

Script hanya dapat berjalan apabila:

- final copy V3 berstatus `COMPLETED`;
- `blocking_failures` kosong;
- manifest final berisi 594 sampel;
- semua pemeriksaan missing/duplicate/non-validated bernilai nol;
- folder V3 `Preprocessed_594` dan `ROI_594` masing-masing lengkap.

Rilis menggunakan `sample_id` sebagai filename dan menyimpan SHA-256 untuk
raw, preprocessed, ROI, serta seluruh file dokumentasi.


In [ ]:
%pip install pandas


In [ ]:
from __future__ import annotations

import hashlib
import json
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

# Folder produced by Production FINAL v3.
V3_OUTPUT_DIR = Path(r"V3_normal_provenance_production_v3")

# Frozen, versioned production release.
RELEASE_DIR = Path(r"AMD_DATASET_594_PRODUCTION_v1")

# Refuse to overwrite an existing release.
FAIL_IF_RELEASE_EXISTS = True

# Optional ZIP archive. Keep False initially because the image collection
# can be large. The folder itself is already the canonical release.
CREATE_ZIP_ARCHIVE = False


# ============================================================
# REQUIRED V3 ARTIFACTS
# ============================================================

REQUIRED_FILES = [
    "pipeline_provenance_manifest_594.csv",
    "pipeline_provenance_manifest_597.csv",
    "provenance_integrity_checks.csv",
    "run_metadata.json",
    "manual_validation_log.csv",
    "VALIDATION_REPORT.md",
    "copy_log_594.csv",
    "normal_provenance_mapping_order_aligned.csv",
    "normal_preprocessed_to_roi_order_alignment.csv",
    "raw_structure_b_audit.csv",
]

OPTIONAL_AUDIT_FILES = [
    "final_ids_not_in_master.csv",
    "master_ids_excluded_from_final.csv",
    "outer_fold_ids_not_in_final.csv",
    "final_ids_without_outer_fold.csv",
    "duplicate_filename_within_class.csv",
    "missing_raw_structure_b.csv",
    "normal_raw_to_preprocessed_top_candidates.csv",
]


# ============================================================
# HELPERS
# ============================================================

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def canonical_text(value: object) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip()


def resolve_existing_path(value: object, label: str) -> Path:
    path = Path(canonical_text(value))
    if not path.exists() or not path.is_file():
        raise FileNotFoundError(f"{label} does not exist: {path}")
    return path


def copy_and_hash(source: Path, destination: Path) -> dict:
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)

    source_hash = sha256_file(source)
    destination_hash = sha256_file(destination)

    if source_hash != destination_hash:
        raise RuntimeError(
            "Checksum mismatch after copy:\n"
            f"source={source}\n"
            f"destination={destination}"
        )

    return {
        "source_path": str(source),
        "destination_path": str(destination),
        "sha256": destination_hash,
        "size_bytes": destination.stat().st_size,
    }


def load_and_validate_v3() -> tuple[pd.DataFrame, dict, pd.DataFrame]:
    if not V3_OUTPUT_DIR.exists():
        raise FileNotFoundError(
            f"V3 output directory does not exist: {V3_OUTPUT_DIR}"
        )

    missing = [
        name for name in REQUIRED_FILES
        if not (V3_OUTPUT_DIR / name).exists()
    ]
    if missing:
        raise FileNotFoundError(
            "Required V3 artifacts are missing:\n- "
            + "\n- ".join(missing)
        )

    manifest = pd.read_csv(
        V3_OUTPUT_DIR / "pipeline_provenance_manifest_594.csv"
    )
    metadata = json.loads(
        (V3_OUTPUT_DIR / "run_metadata.json").read_text(encoding="utf-8")
    )
    checks = pd.read_csv(
        V3_OUTPUT_DIR / "provenance_integrity_checks.csv"
    )

    required_columns = {
        "sample_id",
        "class_label",
        "source_dataset",
        "original_filename",
        "raw_path",
        "preprocessed_path_597",
        "roi_path_597",
        "mapping_status",
        "outer_fold",
    }
    absent = sorted(required_columns - set(manifest.columns))
    if absent:
        raise ValueError(
            "The 594 manifest is missing required columns: "
            + ", ".join(absent)
        )

    manifest["sample_id"] = (
        manifest["sample_id"].astype(str).str.strip()
    )

    if len(manifest) != 594:
        raise ValueError(
            f"Expected 594 rows in final manifest; found {len(manifest)}."
        )
    if manifest["sample_id"].duplicated().any():
        raise ValueError("Duplicate sample_id found in final manifest.")
    if manifest["sample_id"].eq("").any():
        raise ValueError("Blank sample_id found in final manifest.")

    accepted = {"AUTO_CONFIRMED", "MANUALLY_VERIFIED", "CONFIRMED"}
    invalid = sorted(
        set(manifest["mapping_status"].dropna().astype(str)) - accepted
    )
    if invalid:
        raise ValueError(
            "Unaccepted mapping statuses found: " + ", ".join(invalid)
        )

    blocking = metadata.get("blocking_failures", [])
    if blocking:
        raise ValueError(
            "V3 still contains blocking failures:\n- "
            + "\n- ".join(map(str, blocking))
        )

    if metadata.get("copy_status") != "COMPLETED":
        raise ValueError(
            "V3 final copy status is not COMPLETED: "
            f"{metadata.get('copy_status')}"
        )

    # Validate the specific integrity checks that must equal zero.
    check_map = dict(
        zip(checks["check"].astype(str), checks["value"])
    )
    zero_checks = [
        "missing_preprocessed_path_597",
        "missing_roi_path_597",
        "duplicate_preprocessed_assignments_597",
        "duplicate_roi_assignments_597",
        "missing_outer_fold_594",
        "non_validated_mapping_594",
    ]
    bad_zero_checks = {
        name: check_map.get(name)
        for name in zero_checks
        if str(check_map.get(name)) not in {"0", "0.0"}
    }
    if bad_zero_checks:
        raise ValueError(
            "Integrity checks expected to be zero but were not:\n"
            + json.dumps(bad_zero_checks, indent=2)
        )

    return manifest.sort_values("sample_id").reset_index(drop=True), metadata, checks


def build_release() -> None:
    manifest, metadata, checks = load_and_validate_v3()

    if RELEASE_DIR.exists():
        if FAIL_IF_RELEASE_EXISTS:
            raise FileExistsError(
                f"Release already exists and will not be overwritten: "
                f"{RELEASE_DIR}"
            )
        shutil.rmtree(RELEASE_DIR)

    raw_dir = RELEASE_DIR / "RAW_594"
    pre_dir = RELEASE_DIR / "PREPROCESSED_594"
    roi_dir = RELEASE_DIR / "ROI_594"
    provenance_dir = RELEASE_DIR / "provenance"

    for directory in [raw_dir, pre_dir, roi_dir, provenance_dir]:
        directory.mkdir(parents=True, exist_ok=True)

    records = []

    for index, row in manifest.iterrows():
        sample_id = canonical_text(row["sample_id"])
        class_label = canonical_text(row["class_label"])

        raw_source = resolve_existing_path(row["raw_path"], "Raw source")

        # Prefer the immutable V3 copies for preprocessed and ROI.
        pre_candidates = list(
            (V3_OUTPUT_DIR / "Preprocessed_594").glob(f"{sample_id}.*")
        )
        roi_candidates = list(
            (V3_OUTPUT_DIR / "ROI_594").glob(f"{sample_id}.*")
        )

        if len(pre_candidates) != 1:
            raise ValueError(
                f"Expected exactly one V3 preprocessed copy for {sample_id}; "
                f"found {len(pre_candidates)}."
            )
        if len(roi_candidates) != 1:
            raise ValueError(
                f"Expected exactly one V3 ROI copy for {sample_id}; "
                f"found {len(roi_candidates)}."
            )

        pre_source = pre_candidates[0]
        roi_source = roi_candidates[0]

        raw_destination = (
            raw_dir / class_label
            / f"{sample_id}{raw_source.suffix.lower()}"
        )
        pre_destination = (
            pre_dir / class_label
            / f"{sample_id}{pre_source.suffix.lower()}"
        )
        roi_destination = (
            roi_dir / class_label
            / f"{sample_id}{roi_source.suffix.lower()}"
        )

        raw_info = copy_and_hash(raw_source, raw_destination)
        pre_info = copy_and_hash(pre_source, pre_destination)
        roi_info = copy_and_hash(roi_source, roi_destination)

        records.append(
            {
                "sample_id": sample_id,
                "source_dataset": row["source_dataset"],
                "class_label": class_label,
                "original_filename": row["original_filename"],
                "outer_fold": row["outer_fold"],
                "mapping_status": row["mapping_status"],
                "raw_release_path": str(
                    raw_destination.relative_to(RELEASE_DIR)
                ),
                "raw_sha256": raw_info["sha256"],
                "raw_size_bytes": raw_info["size_bytes"],
                "preprocessed_release_path": str(
                    pre_destination.relative_to(RELEASE_DIR)
                ),
                "preprocessed_sha256": pre_info["sha256"],
                "preprocessed_size_bytes": pre_info["size_bytes"],
                "roi_release_path": str(
                    roi_destination.relative_to(RELEASE_DIR)
                ),
                "roi_sha256": roi_info["sha256"],
                "roi_size_bytes": roi_info["size_bytes"],
            }
        )

        if (index + 1) % 50 == 0 or (index + 1) == len(manifest):
            print(f"Copied and verified {index + 1}/{len(manifest)} samples")

    release_manifest = pd.DataFrame(records)
    release_manifest.to_csv(
        provenance_dir / "release_manifest_594_sha256.csv",
        index=False,
        encoding="utf-8-sig",
    )

    # Independent outer-fold file derived from the validated final manifest.
    manifest[["sample_id", "outer_fold"]].to_csv(
        RELEASE_DIR / "outer_fold_assignments.csv",
        index=False,
        encoding="utf-8-sig",
    )

    # Preserve the original final provenance manifest unchanged.
    for name in REQUIRED_FILES + OPTIONAL_AUDIT_FILES:
        source = V3_OUTPUT_DIR / name
        if source.exists():
            shutil.copy2(source, provenance_dir / name)

    # Create checksums for every non-image release artifact as well.
    artifact_records = []
    for path in sorted(RELEASE_DIR.rglob("*")):
        if path.is_file() and path.name != "RELEASE_FILE_CHECKSUMS_SHA256.csv":
            artifact_records.append(
                {
                    "relative_path": str(path.relative_to(RELEASE_DIR)),
                    "sha256": sha256_file(path),
                    "size_bytes": path.stat().st_size,
                }
            )

    pd.DataFrame(artifact_records).to_csv(
        RELEASE_DIR / "RELEASE_FILE_CHECKSUMS_SHA256.csv",
        index=False,
        encoding="utf-8-sig",
    )

    summary = {
        "release_name": RELEASE_DIR.name,
        "release_created_utc": datetime.now(timezone.utc).isoformat(),
        "release_status": "FROZEN",
        "source_pipeline_version": metadata.get(
            "pipeline_version", "Production FINAL v3"
        ),
        "sample_count": len(release_manifest),
        "raw_file_count": len(list(raw_dir.rglob("*.*"))),
        "preprocessed_file_count": len(list(pre_dir.rglob("*.*"))),
        "roi_file_count": len(list(roi_dir.rglob("*.*"))),
        "class_counts": (
            release_manifest["class_label"].value_counts().to_dict()
        ),
        "mapping_status_counts": (
            release_manifest["mapping_status"].value_counts().to_dict()
        ),
        "blocking_failures": [],
        "immutability_note": (
            "Do not modify files in this release. Create a new version "
            "directory for any future change."
        ),
    }

    (RELEASE_DIR / "RELEASE_METADATA.json").write_text(
        json.dumps(summary, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    readme = f"""# AMD Dataset 594 — Production v1

## Release status

**FROZEN**

This directory is the immutable production release reconstructed and validated
from the 597-sample master collection. Three exact duplicate images were
excluded, leaving 594 retained unique samples.

## Contents

- `RAW_594/`: raw images renamed by immutable `sample_id`
- `PREPROCESSED_594/`: validated preprocessed images
- `ROI_594/`: validated ROI images
- `outer_fold_assignments.csv`: outer-fold assignment for all 594 samples
- `provenance/release_manifest_594_sha256.csv`: sample-level paths and SHA-256
- `RELEASE_FILE_CHECKSUMS_SHA256.csv`: checksum inventory for the release
- `RELEASE_METADATA.json`: release summary and version metadata
- `provenance/`: original V3 manifests, audit records, and validation reports

## Validation summary

- Final retained samples: 594
- NORMAL mappings: 287
- Automatically confirmed NORMAL mappings: 243
- Manually visually verified NORMAL mappings: 44
- Unresolved NORMAL mappings: 0
- Missing preprocessed paths: 0
- Missing ROI paths: 0
- Duplicate preprocessed assignments: 0
- Duplicate ROI assignments: 0
- Missing outer-fold assignments: 0
- Blocking failures: 0

## Immutability rule

Do not rename, replace, edit, add, or delete files inside this directory.

Any future correction, transformation, or experiment-specific derivative must
be created under a new version, for example:

`AMD_DATASET_594_PRODUCTION_v2`

The `sample_id` values and the original outer-fold assignments must remain
unchanged unless a formally documented correction requires a new release.
"""

    (RELEASE_DIR / "README.md").write_text(readme, encoding="utf-8")

    # Recompute checksum inventory after README and metadata were added.
    artifact_records = []
    for path in sorted(RELEASE_DIR.rglob("*")):
        if path.is_file() and path.name != "RELEASE_FILE_CHECKSUMS_SHA256.csv":
            artifact_records.append(
                {
                    "relative_path": str(path.relative_to(RELEASE_DIR)),
                    "sha256": sha256_file(path),
                    "size_bytes": path.stat().st_size,
                }
            )
    pd.DataFrame(artifact_records).to_csv(
        RELEASE_DIR / "RELEASE_FILE_CHECKSUMS_SHA256.csv",
        index=False,
        encoding="utf-8-sig",
    )

    # Final count validation.
    counts = {
        "RAW_594": len(list(raw_dir.rglob("*.*"))),
        "PREPROCESSED_594": len(list(pre_dir.rglob("*.*"))),
        "ROI_594": len(list(roi_dir.rglob("*.*"))),
        "release_manifest": len(release_manifest),
        "outer_fold_assignments": len(
            pd.read_csv(RELEASE_DIR / "outer_fold_assignments.csv")
        ),
    }

    expected = {
        "RAW_594": 594,
        "PREPROCESSED_594": 594,
        "ROI_594": 594,
        "release_manifest": 594,
        "outer_fold_assignments": 594,
    }

    if counts != expected:
        raise RuntimeError(
            "Final release count validation failed:\n"
            f"observed={counts}\nexpected={expected}"
        )

    if CREATE_ZIP_ARCHIVE:
        archive = shutil.make_archive(
            str(RELEASE_DIR),
            "zip",
            root_dir=RELEASE_DIR.parent,
            base_dir=RELEASE_DIR.name,
        )
        print(f"ZIP archive created: {archive}")

    print("\n=== FROZEN RELEASE COMPLETED ===")
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    print(f"Release directory: {RELEASE_DIR.resolve()}")


if __name__ == "__main__":
    build_release()
